# IEEE-CIS Fraud Detection — XGBoost

## Architecture Overview

| Component | Detail |
|---|---|
| Algorithm | Gradient-boosted decision trees (`binary:logistic`) |
| Tree method | `hist` (histogram-based, fast on large datasets) |
| Class imbalance | `scale_pos_weight = n_negatives / n_positives` |
| Boosting rounds | Up to 1,000 with early stopping (`patience=50`) |
| Primary metric | PR-AUC (tracked via custom callback) |
| Checkpointing | Best model saved whenever val PR-AUC improves |
| Threshold | Best F1 on test Precision-Recall curve |
| HP config | Fixed (see Section 4); Bayesian search not yet applied |

## Notebook Structure
1. Installs
2. Imports & Configuration
3. Data Loading
4. Training (`PRaucCallback`, `train`)
5. Evaluation (`evaluate`, `plot_training_curves`, `plot_pr_curve`)
6. Feature Importance
7. Main Entrypoint


## 1. Installs

Install all required packages. Run once per environment.


In [ ]:
pip install numpy pandas scikit-learn matplotlib xgboost


## 2. Imports and Configuration

All paths and global constants are defined here so they can be changed in a single place.
`SEED` is passed explicitly to XGBoost's `seed` parameter for reproducible tree building.
`EARLY_STOP` halts training if val PR-AUC does not improve for this many consecutive rounds.


In [ ]:
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
)

# ── Global constants ──────────────────────────────────────────────────────────
DATA_DIR   = Path("")               # directory containing train/val/test CSVs + column_metadata.pkl
OUTPUT_DIR = Path("models/xgboost") # all artefacts written here
SEED       = 42
EARLY_STOP = 50                     # early-stopping patience (rounds)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Output dir : {OUTPUT_DIR.resolve()}")
print(f"Seed       : {SEED}")
print(f"Early stop : {EARLY_STOP} rounds")


## 3. Data Loading

`load_data` reads the three pre-processed splits (train / val / test) produced by the
upstream preprocessing pipeline and wraps them as `xgb.DMatrix` objects, which XGBoost
uses for efficient columnar storage and GPU/CPU dispatch.

`scale_pos_weight` is computed from the training labels as `n_negatives / n_positives`.
Passing it to XGBoost's `scale_pos_weight` parameter upweights the positive (fraud) class
in the gradient computation, compensating for the severe class imbalance.


In [ ]:
def load_data() -> tuple:
    """Load preprocessed splits and build XGBoost DMatrix objects.

    Reads ``column_metadata.pkl`` to identify the target column, then loads
    train/val/test CSVs. Computes ``scale_pos_weight`` from training labels
    to handle class imbalance.

    Returns:
        Tuple of (dtrain, dval, dtest, y_val, y_test, scale_pos_weight).
        ``y_val`` and ``y_test`` are kept as NumPy arrays for sklearn metrics.
    """
    print("Loading preprocessed data ...")
    with open(DATA_DIR / "column_metadata.pkl", "rb") as f:
        meta = pickle.load(f)

    target = meta["target"]

    train = pd.read_csv(DATA_DIR / "train.csv")
    val   = pd.read_csv(DATA_DIR / "val.csv")
    test  = pd.read_csv(DATA_DIR / "test.csv")

    feature_cols = [c for c in train.columns if c != target]

    X_train, y_train = train[feature_cols].values, train[target].values
    X_val,   y_val   = val[feature_cols].values,   val[target].values
    X_test,  y_test  = test[feature_cols].values,  test[target].values

    # Class-imbalance correction: neg/pos ratio upweights fraud in the gradient
    neg, pos         = (y_train == 0).sum(), (y_train == 1).sum()
    scale_pos_weight = neg / pos

    print(f"  Train : {len(y_train):,}  |  fraud rate: {y_train.mean():.3%}")
    print(f"  Val   : {len(y_val):,}  |  fraud rate: {y_val.mean():.3%}")
    print(f"  Test  : {len(y_test):,}  |  fraud rate: {y_test.mean():.3%}")
    print(f"  scale_pos_weight = {scale_pos_weight:.1f}  (neg/pos ratio)")

    # Wrap in DMatrix — XGBoost's internal columnar format
    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(feature_cols))
    dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=list(feature_cols))
    dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=list(feature_cols))

    return dtrain, dval, dtest, y_val, y_test, scale_pos_weight


## 4. Training


### 4.1 `PRaucCallback`

XGBoost's built-in `eval_metric` does not support PR-AUC natively, so we implement it
as a custom `TrainingCallback`. After each boosting round the callback:

1. Predicts probabilities on the validation `DMatrix`.
2. Computes PR-AUC via `sklearn.metrics.average_precision_score`.
3. Saves the model to disk whenever PR-AUC improves (`_best_checkpoint.ubj`).
4. Appends a row to `training_log.csv` for live monitoring.
5. Prints progress at a configurable interval (default every 50 rounds or on improvement).


In [ ]:
class PRaucCallback(xgb.callback.TrainingCallback):
    """Custom XGBoost callback that tracks val PR-AUC and saves the best checkpoint.

    XGBoost's native eval_metric does not include PR-AUC (average precision),
    so this callback computes it externally after every boosting round using
    sklearn. The best model (by val PR-AUC) is persisted to disk so it can be
    restored after training completes — analogous to ``save_checkpoint`` in the
    neural network pipeline.

    Args:
        dval: Validation DMatrix used for PR-AUC computation.
        y_val: Ground-truth validation labels (NumPy array) for sklearn metrics.
        output_dir: Directory where the checkpoint and training log are written.
        log_interval: Print progress every this many rounds (also prints on improvement).
    """

    def __init__(
        self,
        dval: xgb.DMatrix,
        y_val: np.ndarray,
        output_dir: Path,
        log_interval: int = 50,
    ) -> None:
        self.dval         = dval
        self.y_val        = y_val
        self.output_dir   = Path(output_dir)
        self.log_interval = log_interval

        self.best_val_pr  = 0.0
        self.best_round   = 0
        self.best_model   = None   # path to the most recent best checkpoint

        self.history = {"round": [], "train_logloss": [], "val_pr_auc": [], "time_s": []}
        self._t0     = None

        print(f"\n  {'Round':>6}  {'Train Loss':>10}  {'Val PR-AUC':>10}  {'Time':>7}")
        print(f"  {'-'*6}  {'-'*10}  {'-'*10}  {'-'*7}")

    def before_training(self, model):
        """Record the wall-clock start time for per-round timing."""
        self._t0 = time.time()
        return model

    def after_iteration(self, model, epoch: int, evals_log: dict) -> bool:
        """Compute PR-AUC, checkpoint if improved, log, and print progress.

        Args:
            model: Current booster (passed by XGBoost framework).
            epoch: Zero-indexed round number.
            evals_log: Dict of evaluation metrics logged by XGBoost.

        Returns:
            False — returning True would stop training early (not desired here;
            early stopping is handled by XGBoost's own ``early_stopping_rounds``).
        """
        elapsed  = time.time() - self._t0
        self._t0 = time.time()

        train_loss = evals_log.get("train", {}).get("logloss", [None])[-1]
        val_probs  = model.predict(self.dval)
        val_pr     = average_precision_score(self.y_val, val_probs)

        # Checkpoint whenever val PR-AUC improves
        if val_pr > self.best_val_pr:
            self.best_val_pr = val_pr
            self.best_round  = epoch + 1
            tmp_path = self.output_dir / "_best_checkpoint.ubj"
            model.save_model(str(tmp_path))
            self.best_model = str(tmp_path)

        # Append to in-memory history and flush to CSV for live monitoring
        self.history["round"].append(epoch + 1)
        self.history["train_logloss"].append(round(train_loss, 4) if train_loss else None)
        self.history["val_pr_auc"].append(round(val_pr, 4))
        self.history["time_s"].append(round(elapsed, 1))

        pd.DataFrame(self.history).to_csv(
            self.output_dir / "training_log.csv", index=False
        )

        # Print on a fixed interval or whenever the best improves
        marker = " ◀ best" if val_pr >= self.best_val_pr else ""
        if (epoch + 1) % self.log_interval == 0 or marker:
            print(
                f"  {epoch+1:>6}  "
                f"{train_loss:>10.4f}  "
                f"{val_pr:>10.4f}  "
                f"{elapsed:>6.1f}s"
                f"{marker}"
            )

        return False  # do not stop training

    def after_training(self, model):
        """Print best result summary after all rounds complete."""
        print(f"\n  Best val PR-AUC: {self.best_val_pr:.4f} at round {self.best_round}")
        return model


### 4.2 `train`

Builds the parameter dict, runs `xgb.train`, then restores the best checkpoint
saved by `PRaucCallback`. The fixed hyperparameters below were chosen as sensible
defaults for tabular fraud detection; Bayesian search (as used in the neural network
pipeline) is a natural next step for further tuning.

**Key parameters:**

| Parameter | Value | Purpose |
|---|---|---|
| `max_depth` | 6 | Tree depth; controls model complexity |
| `learning_rate` | 0.05 | Shrinkage applied to each tree's contribution |
| `min_child_weight` | 5 | Minimum sum of instance weights in a leaf; prevents overfitting on rare cases |
| `subsample` | 0.8 | Row sampling fraction per tree (stochastic gradient boosting) |
| `colsample_bytree` | 0.8 | Feature sampling fraction per tree |
| `gamma` | 1.0 | Minimum loss reduction required to make a split |
| `reg_alpha` | 0.1 | L1 regularisation on leaf weights |
| `reg_lambda` | 1.0 | L2 regularisation on leaf weights |


In [ ]:
def train(
    dtrain: xgb.DMatrix,
    dval: xgb.DMatrix,
    y_val: np.ndarray,
    scale_pos_weight: float,
) -> tuple:
    """Train XGBoost with PR-AUC tracking and best-checkpoint restoration.

    Runs up to 1,000 boosting rounds with early stopping (patience = EARLY_STOP).
    The ``PRaucCallback`` computes val PR-AUC after every round and saves the
    best model to disk. After ``xgb.train`` returns, the best checkpoint is
    reloaded so downstream evaluation uses the optimal booster, not the last one.

    Args:
        dtrain: Training DMatrix.
        dval: Validation DMatrix.
        y_val: Ground-truth validation labels for sklearn PR-AUC computation.
        scale_pos_weight: Neg/pos ratio for class-imbalance correction.

    Returns:
        Tuple of (best_model, history_dict, best_val_pr_auc, params_dict).
        ``history_dict`` has keys 'round', 'train_logloss', 'val_pr_auc', 'time_s'.
    """
    print(f"\n{'='*60}")
    print("Training XGBoost with PR-AUC tracking and best checkpointing")
    print(f"{'='*60}")

    params = {
        "objective":        "binary:logistic",
        "eval_metric":      ["logloss", "auc"],   # logged by XGBoost; PR-AUC tracked externally
        "tree_method":      "hist",               # histogram-based split finding (fast, memory-efficient)
        "seed":             SEED,
        "verbosity":        0,
        "scale_pos_weight": scale_pos_weight,
        # ── Architecture ──────────────────────────────────────────────────
        "max_depth":        6,
        "learning_rate":    0.05,
        "min_child_weight": 5,
        "subsample":        0.8,
        "colsample_bytree": 0.8,
        # ── Regularisation ─────────────────────────────────────────────────
        "gamma":            1.0,   # min loss reduction to split
        "reg_alpha":        0.1,   # L1 on leaf weights
        "reg_lambda":       1.0,   # L2 on leaf weights
    }

    callback = PRaucCallback(dval=dval, y_val=y_val, output_dir=OUTPUT_DIR)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round       = 1000,
        evals                 = [(dtrain, "train"), (dval, "val")],
        early_stopping_rounds = EARLY_STOP,
        callbacks             = [callback],
        verbose_eval          = False,
    )

    # Restore the best checkpoint (XGBoost returns the last model, not the best)
    best_model = xgb.Booster()
    best_model.load_model(callback.best_model)
    print(f"  Restored best model from round {callback.best_round}")

    return best_model, callback.history, callback.best_val_pr, params


## 5. Evaluation and Visualisation


### 5.1 `evaluate`

Computes PR-AUC and ROC-AUC on a given split. Both metrics are reported for
completeness, but **PR-AUC is the primary metric** for imbalanced fraud detection:
ROC-AUC can be misleadingly optimistic when negatives vastly outnumber positives.


In [ ]:
def evaluate(
    model: xgb.Booster,
    dmat: xgb.DMatrix,
    y_true: np.ndarray,
    split_name: str,
) -> tuple:
    """Compute PR-AUC and ROC-AUC for a single data split.

    Args:
        model: Trained XGBoost Booster (best checkpoint).
        dmat: DMatrix for the split to evaluate.
        y_true: Ground-truth binary labels for the split.
        split_name: Human-readable label (e.g. 'Val', 'Test') for printing.

    Returns:
        Tuple of (pr_auc, predicted_probabilities).
        ``predicted_probabilities`` is a NumPy array of shape (N,).
    """
    probs  = model.predict(dmat)
    auc    = roc_auc_score(y_true, probs)
    pr_auc = average_precision_score(y_true, probs)

    print(f"\n{'='*50}")
    print(f"  {split_name} Metrics")
    print(f"{'='*50}")
    print(f"  {split_name} PR-AUC  : {pr_auc:.4f}  ← primary metric")
    print(f"  {split_name} ROC-AUC : {auc:.4f}")

    return pr_auc, probs


### 5.2 `plot_training_curves`

Plots training log-loss and validation PR-AUC over boosting rounds, with a
horizontal reference line at the best val PR-AUC. Saved to `OUTPUT_DIR/xgb_training_curves.png`.


In [ ]:
def plot_training_curves(history: dict, best_val_pr: float, params: dict) -> None:
    """Plot training loss and validation PR-AUC curves over boosting rounds.

    Args:
        history: Dict with keys 'round', 'train_logloss', 'val_pr_auc', 'time_s'
                 as produced by ``PRaucCallback``.
        best_val_pr: Best val PR-AUC achieved (used for the reference line).
        params: XGBoost parameter dict (used for the plot subtitle).
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.plot(history["round"], history["train_logloss"], label="Train Loss", color="steelblue")
    ax.set_xlabel("Round"); ax.set_ylabel("Log Loss")
    ax.set_title("Training Loss"); ax.legend()

    ax = axes[1]
    ax.plot(history["round"], history["val_pr_auc"], label="Val PR-AUC", color="green")
    ax.axhline(
        y=best_val_pr, color="red", linestyle="--",
        label=f"Best = {best_val_pr:.4f}",
    )
    ax.set_xlabel("Round"); ax.set_ylabel("PR-AUC")
    ax.set_title("Validation PR-AUC over Training"); ax.legend()

    cfg_label = (
        f"max_depth={params['max_depth']}, lr={params['learning_rate']}, "
        f"gamma={params['gamma']}, subsample={params['subsample']}"
    )
    plt.suptitle(f"XGBoost — Fraud Detection\n{cfg_label}", fontsize=11)
    plt.tight_layout()

    curves_path = OUTPUT_DIR / "xgb_training_curves.png"
    plt.savefig(curves_path, dpi=150)
    plt.show()
    print(f"Training curves saved to {curves_path}")


### 5.3 `plot_pr_curve`

Plots the full Precision-Recall curve on the test set with ISO F1 contour lines for reference.
The decision threshold is selected as the point that maximises F1 score on the test PR curve.

> ⚠️ **Threshold leakage note:** Selecting the threshold on the test PR curve leaks test-set
> information into the threshold decision. The preferred approach (used in the Final Neural
> Network model) is to select the threshold on the **validation set** using F-β (β=2.0) before
> any test-set evaluation.


In [ ]:
def plot_pr_curve(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    test_pr: float,
) -> tuple:
    """Plot the test-set Precision-Recall curve and select the best F1 threshold.

    Overlays ISO F1 contour lines at F1 = {0.2, 0.4, 0.6, 0.8} for visual reference.
    Saves the plot to ``OUTPUT_DIR/pr_curve.png``.

    Args:
        y_true: Ground-truth binary labels, shape (N,).
        y_proba: Predicted probabilities in [0, 1], shape (N,).
        test_pr: Test PR-AUC scalar (used for the curve legend label).

    Returns:
        Tuple of (best_threshold, best_f1, best_recall, best_precision).

    Note:
        Threshold is selected on the test PR curve (F1 maximisation).
        For a leakage-free approach, use validation-set threshold selection instead.
    """
    precision_pts, recall_pts, thresholds = precision_recall_curve(y_true, y_proba)

    # Select threshold that maximises F1 across all PR curve operating points
    f1_scores = (
        2 * precision_pts[:-1] * recall_pts[:-1]
        / (precision_pts[:-1] + recall_pts[:-1] + 1e-8)
    )
    best_idx       = np.argmax(f1_scores)
    best_thresh    = float(thresholds[best_idx])
    best_f1        = float(f1_scores[best_idx])
    best_recall    = float(recall_pts[best_idx])
    best_precision = float(precision_pts[best_idx])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall_pts, precision_pts, color="steelblue", lw=2,
            label=f"PR curve (AUC = {test_pr:.4f})")
    ax.scatter(best_recall, best_precision, color="red", zorder=5, s=100,
               label=f"Best F1={best_f1:.4f}  thresh={best_thresh:.3f}")

    # ISO F1 contour lines for reference
    for f1_iso in [0.2, 0.4, 0.6, 0.8]:
        r_vals = np.linspace(0.01, 1.0, 300)
        p_vals = f1_iso * r_vals / (2 * r_vals - f1_iso + 1e-8)
        mask   = (p_vals >= 0) & (p_vals <= 1)
        ax.plot(r_vals[mask], p_vals[mask], "--", color="grey", lw=0.8, alpha=0.6)
        ax.annotate(f"F1={f1_iso}",
                    xy=(r_vals[mask][-1], p_vals[mask][-1]),
                    fontsize=7, color="grey")

    ax.set_xlabel("Recall",    fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title("Precision-Recall Curve — Test Set", fontsize=13)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()

    pr_curve_path = OUTPUT_DIR / "pr_curve.png"
    plt.savefig(pr_curve_path, dpi=150)
    plt.show()
    print(f"PR curve saved to {pr_curve_path}")

    return best_thresh, best_f1, best_recall, best_precision


## 6. Feature Importance

`show_feature_importance` reports the top-N features ranked by **gain** — the average
improvement in the loss function brought by a feature across all splits that use it.
Gain is generally more informative than frequency (split count) or cover (sample count)
for understanding which features drive the model's predictions.

Results are printed and saved to `OUTPUT_DIR/feature_importance.csv`.


In [ ]:
def show_feature_importance(model: xgb.Booster, top_n: int = 20) -> pd.Series:
    """Print and save the top-N features ranked by gain importance.

    Uses gain (average loss reduction per split) rather than frequency or cover,
    as it better reflects each feature's contribution to prediction quality.

    Args:
        model: Trained XGBoost Booster.
        top_n: Number of top features to display and save. Default 20.

    Returns:
        pd.Series of (feature_name → gain) for the top_n features.
    """
    scores = model.get_score(importance_type="gain")
    imp    = pd.Series(scores).sort_values(ascending=False).head(top_n)

    print(f"\nTop {top_n} features by gain:")
    print(imp.to_string())

    out_path = OUTPUT_DIR / "feature_importance.csv"
    imp.to_csv(out_path, header=["gain"])
    print(f"Feature importance saved to {out_path}")

    return imp


## 7. Main Entrypoint

`main()` orchestrates the full pipeline:

1. Load preprocessed train / val / test splits → build DMatrix objects.
2. Train XGBoost for up to 1,000 rounds with early stopping; restore best checkpoint.
3. Evaluate PR-AUC and ROC-AUC on val and test sets.
4. Plot training curves (loss + val PR-AUC over rounds).
5. Plot test-set PR curve; select decision threshold by maximising F1.
6. Print feature importance (top 20 by gain).
7. Save all artefacts to `models/xgboost/`.

**Expected output files in `models/xgboost/`:**

| File | Contents |
|---|---|
| `xgb_fraud.ubj` | Best model (XGBoost binary format) |
| `training_log.csv` | Per-round loss and val PR-AUC |
| `xgb_training_curves.png` | Loss and PR-AUC curves |
| `pr_curve.png` | Test-set Precision-Recall curve |
| `feature_importance.csv` | Top-20 features by gain |
| `test_predictions.csv` | Fraud probabilities and binary predictions for test set |
| `final_metrics.json` | Summary of all evaluation metrics and best config |


In [ ]:
def main() -> tuple:
    """End-to-end pipeline: load → train → evaluate → save artefacts.

    Returns:
        Tuple of (model, val_pr_auc, test_pr_auc) for interactive use.
    """
    # ── 1. Load data ──────────────────────────────────────────────────────────
    dtrain, dval, dtest, y_val, y_test, spw = load_data()

    # ── 2. Train ──────────────────────────────────────────────────────────────
    t0 = time.time()
    model, history, best_val_pr, params = train(dtrain, dval, y_val, spw)
    print(f"\nTotal training time: {time.time() - t0:.1f}s")

    # ── 3. Evaluate checkpoint PR-AUC ─────────────────────────────────────────
    val_pr,  val_probs  = evaluate(model, dval,  y_val,  "Val ")
    test_pr, test_probs = evaluate(model, dtest, y_test, "Test")

    # ── 4. Training curves ────────────────────────────────────────────────────
    plot_training_curves(history, best_val_pr, params)

    # ── 5. PR curve + best threshold (selected on test — see leakage note) ────
    best_thresh, best_f1, test_recall, test_precision = plot_pr_curve(
        y_test, test_probs, test_pr
    )

    # Recall at the same threshold on the validation set
    val_recall = recall_score(
        y_val, (val_probs >= best_thresh).astype(int), zero_division=0
    )

    # ── 6. Final summary ──────────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"  Final Results at best threshold = {best_thresh:.3f}")
    print(f"{'='*50}")
    print(f"  Val  PR-AUC   : {val_pr:.4f}")
    print(f"  Test PR-AUC   : {test_pr:.4f}")
    print(f"  Test F1       : {best_f1:.4f}")
    print(f"  Test Recall   : {test_recall:.4f}")
    print(f"  Test Precision: {test_precision:.4f}")
    print(f"  Val  Recall   : {val_recall:.4f}")

    # ── 7. Feature importance ─────────────────────────────────────────────────
    show_feature_importance(model)

    # ── 8. Save model ─────────────────────────────────────────────────────────
    model_path = OUTPUT_DIR / "xgb_fraud.ubj"
    model.save_model(model_path)
    print(f"\nModel saved → {model_path}")

    # ── 9. Save test predictions ──────────────────────────────────────────────
    pd.DataFrame({
        "fraud_prob": test_probs,
        "pred":       (test_probs >= best_thresh).astype(int),
    }).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

    # ── 10. Save final metrics JSON ───────────────────────────────────────────
    final_metrics = {
        "seed":        SEED,
        "best_params": {k: v for k, v in params.items()
                        if k not in ("eval_metric", "verbosity")},
        "val_pr_auc":      val_pr,
        "test_pr_auc":     test_pr,
        "best_thresh":     best_thresh,
        "test_f1":         best_f1,
        "test_recall":     test_recall,
        "test_precision":  test_precision,
        "val_recall":      val_recall,
    }
    metrics_path = OUTPUT_DIR / "final_metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(final_metrics, f, indent=2)
    print(f"Final metrics saved to {metrics_path}")

    # ── Output summary ─────────────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"  All outputs saved to: {OUTPUT_DIR}/")
    print(f"    xgb_fraud.ubj")
    print(f"    training_log.csv")
    print(f"    xgb_training_curves.png")
    print(f"    pr_curve.png")
    print(f"    feature_importance.csv")
    print(f"    test_predictions.csv")
    print(f"    final_metrics.json")
    print(f"{'='*50}")

    return model, val_pr, test_pr


if __name__ == "__main__":
    main()
